# Préparation de l'espace de travail

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
df_lic = pd.read_parquet("data/data_licences/data_licences.parquet")
df_med = pd.read_csv("data/data_medailles/data_medailles_jo.csv")
df = pd.read_parquet("data/data_complet.parquet")

# Fonctions de tracé

In [ ]:
def plot_licences_par_annee(df_lic, sport_code="all", sport_col="code_sport"):
    """
    Affiche un graphique interactif des licences annuelles.

    Paramètres
    ----------
    df_lic : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'licences_annuelles', sport_col].
    sport_code : str ou None, optionnel
        Filtre pour un code de sport spécifique. Si None, affiche tous les sports.
    sport_col : str
        Nom de la colonne contenant le code sport (par défaut 'code_sport').
    output_widget : ipywidgets.Output ou None
        Si fourni, le graphique sera affiché dans ce widget pour permettre une mise à jour interactive.

    Output
    ------
    Graphique interactif Plotly des licences annuelles.
    """
    
    df = df_lic.copy()
    
    # Filtrage par sport si demandé
    if sport_code == "all":
        df = df.copy()
        titre_sport = "tous les sports"
    else:
        df = df[df[sport_col] == sport_code]
        noms_sports = df["sport"].dropna().unique()
        if len(noms_sports) == 1:
            titre_sport = noms_sports[0]
        else:
            titre_sport = ", ".join(noms_sports)

    
    # Agrégation des licences par année
    data = (
        df.groupby("annee")["licences_annuelles"]
          .sum()
          .reset_index()
          .sort_values("annee")
    )
    
    # Calcul de la variation annuelle et du taux d'évolution
    if len(data) >= 2:
        data["variation"] = data["licences_annuelles"].diff()
        data["taux_evolution_%"] = data["licences_annuelles"].pct_change() * 100
    else:
        data["variation"] = np.nan
        data["taux_evolution_%"] = np.nan
        
    # Tracé    
    fig = px.line(
        data, x="annee", y="licences_annuelles",
        title=f"Licences annuelles - {titre_sport}",
        markers=True,
        labels={"annee": "Année", "licences_annuelles": "Licences annuelles"}
    )
    
    fig.update_layout(xaxis=dict(dtick=1))
    fig.show()


In [ ]:
# Code interactif

# Préparation des options du widget, ajout de l'option "all"
sports_names = sorted(df["sport"].dropna().unique())
options = ["all"] + list(sports_names)

# Création du widget
sports_widget = widgets.Dropdown(
    options=options,
    description="Sports :",
    value="all"
)
out = widgets.Output()

# Fonction de mise à jour du graphique lorsqu'on change le sport
def update_graph(change=None):
    """
    Met à jour le graphique interactif lorsque la valeur du widget change.
    """
    clear_output(wait=True)
    display(sports_widget)
    
    selected_sport = sports_widget.value
    
    if selected_sport == "all":
        plot_licences_par_annee(df, sport_code="all", sport_col="code_sport")
    else:
        # Conversion du nom en code_sport
        codes = df.loc[df["sport"] == selected_sport, "code_sport"].unique()
        if len(codes) == 1:
            code = codes[0]
        else:
            code = list(codes)  # si plusieurs codes pour un même nom
        plot_licences_par_annee(df, sport_code=code, sport_col="code_sport")


# Liaison du widget à la fonction de mise à jour
sports_widget.observe(update_graph, names='value')

# Affichage initial
display(sports_widget, out)
update_graph()


In [ ]:
def plot_licences_par_sexe(df_lic, sport_code="all", sport_col="code_sport"):
    """
    Affiche un graphique interactif des licences annuelles par sexe.

    Paramètres
    ----------
    df_lic : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'licences_annuelles', 'sexe', sport_col].
    sport_code : str ou None, optionnel
        Filtre pour un code de sport spécifique. Si None ou "all", affiche tous les sports.
    sport_col : str
        Nom de la colonne contenant le code sport (par défaut 'code_sport').
    output_widget : ipywidgets.Output ou None
        Si fourni, le graphique sera affiché dans ce widget pour permettre une mise à jour interactive.

    Output
    ------
    Graphique interactif Plotly des licences annuelles par sexe.
    """
    
    df = df_lic.copy()
    
    # Filtrage par sport si demandé
    if sport_code == "all":
        df = df.copy()
        titre_sport = "tous les sports"
    else:
        df = df[df[sport_col] == sport_code]
        noms_sports = df["sport"].dropna().unique()
        if len(noms_sports) == 1:
            titre_sport = noms_sports[0]
        else:
            titre_sport = ", ".join(noms_sports)
    
    # Remplacement des valeurs manquantes de "sexe" par 'NR'
    df['sexe'] = df['sexe'].fillna('NR')
    
    # Obtention de toutes les années et de toutes les catégories de sexe
    annees = sorted(df['annee'].unique())
    sexes = df['sexe'].unique()
    
    # Création d'un DataFrame complet avec toutes les combinaisons année x sexe
    complete = pd.MultiIndex.from_product([annees, sexes], names=['annee', 'sexe']).to_frame(index=False)
    
    # Agrégation des licences par année et par sexe
    data = df.groupby(['annee', 'sexe'])['licences_annuelles'].sum().reset_index()
    
    # Merge avec le DataFrame complet pour avoir des 0 si absence de données
    data = complete.merge(data, on=['annee', 'sexe'], how='left').fillna(0)
    
    # Pivot pour le tracé
    pivot = data.pivot(index='annee', columns='sexe', values='licences_annuelles')
    
    # Création du graphique
    fig = go.Figure()
    for sexe in pivot.columns:
        fig.add_trace(go.Scatter(
            x=pivot.index,
            y=pivot[sexe],
            mode='lines+markers',
            name=str(sexe)
        ))
    
    fig.update_layout(
        title=f"Licences annuelles par sexe - {titre_sport}",
        xaxis_title="Année",
        yaxis_title="Licences annuelles",
        xaxis=dict(dtick=1),
        legend_title="Sexe"
    )
    
    fig.show()


In [ ]:
# Code interactif

# Préparation des options du widget, ajout de l'option "all"
sports_names = sorted(df["sport"].dropna().unique())
options = ["all"] + list(sports_names)

# Création du widget
sports_widget = widgets.Dropdown(
    options=options,
    description="Sports :",
    value="all"
)
out = widgets.Output()

# Fonction de mise à jour du graphique lorsqu'on change le sport
def update_graph(change=None):
    """
    Met à jour le graphique interactif lorsque la valeur du widget change.
    """
    clear_output(wait=True)
    display(sports_widget)
    plot_licences_par_sexe(df, sport_code=sports_widget.value)

# Liaison du widget à la fonction de mise à jour
sports_widget.observe(update_graph, names='value')

# Affichage initial
display(sports_widget, out)
update_graph()


In [ ]:
def plot_part_jeunes(df_lic, age_max=15, sport_code="all", sport_col="code_sport"):
    """
    Affiche un graphique interactif Plotly de la part des jeunes licenciés (< age_max) par année.

    Paramètres
    ----------
    df_lic : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'licences_annuelles', 'age', sport_col].
    age_max : int
        Age maximum pour définir les "jeunes" (par défaut 15 ans).
    sport_code : str ou None
        Filtre pour un code de sport spécifique. Si None ou "all", affiche tous les sports.
    sport_col : str
        Nom de la colonne contenant le code sport (par défaut 'code_sport').

    Output
    ------
    Graphique interactif Plotly de la part des jeunes licenciés par année (%).
    """
    df = df_lic.copy()

    # Convertir la colonne 'age' en numérique, valeurs "NR" deviennent NaN
    df['age'] = pd.to_numeric(df['age'], errors='coerce')

    # Filtrage par sport si demandé
    if sport_code == "all":
        df = df.copy()
        titre_sport = "tous les sports"
    else:
        df = df[df[sport_col] == sport_code]
        noms_sports = df["sport"].dropna().unique()
        if len(noms_sports) == 1:
            titre_sport = noms_sports[0]
        else:
            titre_sport = ", ".join(noms_sports)

    # Agrégation des licences par année
    total = df.groupby("annee")["licences_annuelles"].sum().reset_index(name="licences_total")

    # Calcul du total des jeunes (< age_max) par année
    jeunes = df[df["age"] < age_max].groupby("annee")["licences_annuelles"].sum().reset_index(name="licences_jeunes")

    # Fusion des deux tables et calcul de la part des jeunes
    data = jeunes.merge(total, on="annee", how="left")
    data["part_jeunes_%"] = data["licences_jeunes"] / data["licences_total"] * 100

    # Tracé Plotly
    fig = px.line(
        data,
        x="annee",
        y="part_jeunes_%",
        markers=True,
        title=f"Part des licenciés plus jeunes que {age_max} ans - {titre_sport}",
        labels={"annee": "Année", "part_jeunes_%": "Part des jeunes (%)"}
    )

    fig.update_layout(xaxis=dict(dtick=1), width=900, height=500)
    fig.show()


In [ ]:
# Préparation des options du widget, ajout de l'option "all"
sports_names = sorted(df["sport"].dropna().unique())
options = ["all"] + list(sports_names)

# Création du widget Dropdown
sports_widget = widgets.Dropdown(
    options=options,
    description="Sports :",
    value="all"
)
# Création d'un output pour l'affichage du graphique
out = widgets.Output()

# Fonction de mise à jour du graphique lors du changement de valeur du widget
def update_graph(change=None):
    """
    Mise à jour du graphique interactif en fonction du sport sélectionné.
    """
    # Effacement du graphique précédent
    clear_output(wait=True)
    # Affichage du widget
    display(sports_widget)
    
    # Récupération du sport sélectionné
    selected_sport = sports_widget.value
    
    # Cas "all" pour tous les sports
    if selected_sport == "all":
        plot_part_jeunes(df, age_max=15, sport_code="all", sport_col="code_sport")
    else:
        # Conversion du nom du sport en code_sport
        codes = df.loc[df["sport"] == selected_sport, "code_sport"].unique()
        if len(codes) == 1:
            code = codes[0]
        else:
            code = list(codes)  # Plusieurs codes possibles pour un même nom
        plot_part_jeunes(df, age_max=15, sport_code=code, sport_col="code_sport")

# Liaison du widget à la fonction de mise à jour
sports_widget.observe(update_graph, names='value')

# Affichage initial du widget et du graphique
display(sports_widget, out)
update_graph()


In [ ]:
def plot_part_femmes(df_lic, sport_code="HAN", sport_col="code_sport"):
    """
    Affiche un graphique interactif Plotly de la part des femmes parmi les licenciés par année.

    Paramètres
    ----------
    df_lic : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'licences_annuelles', 'sexe', sport_col].
    sport_code : str
        Code du sport à filtrer (par défaut "HAN" pour Handball).
    sport_col : str
        Nom de la colonne contenant le code sport (par défaut 'code_sport').

    Output
    ------
    Graphique interactif Plotly de la part des femmes parmi les licenciés par année.
    """
    df = df_lic.copy()

    # Filtrage par sport si demandé
    if sport_code == "all":
        df = df.copy()
        titre_sport = "tous les sports"
    else:
        df = df[df[sport_col] == sport_code]
        noms_sports = df["sport"].dropna().unique()
        if len(noms_sports) == 1:
            titre_sport = noms_sports[0]
        else:
            titre_sport = ", ".join(noms_sports)

    # Agrégation des licences par année et par sexe
    df_par_sexe = (
        df.groupby(["annee", "sexe"])["licences_annuelles"]
          .sum()
          .reset_index()
          .pivot(index="annee", columns="sexe", values="licences_annuelles")
    )

    # Remplacement des valeurs manquantes par 0 pour éviter les erreurs de division
    df_par_sexe = df_par_sexe.fillna(0)

    # Remplissage des colonnes manquantes
    if 'F' not in df_par_sexe.columns:
        df_par_sexe['F'] = 0
    if 'H' not in df_par_sexe.columns:
        df_par_sexe['H'] = 0

    # Calcul de la part des femmes parmi les licenciés
    df_par_sexe["part_femmes"] = df_par_sexe["F"] / (df_par_sexe['F'] + df_par_sexe['H'])

    # Préparation du graphique Plotly
    fig = px.line(
        df_par_sexe,
        x=df_par_sexe.index,
        y="part_femmes",
        markers=True,
        title=f"Part des femmes parmi les licenciés sportifs - {titre_sport}",
        labels={"x": "Année", "part_femmes": "Part des femmes"}
    )

    # Ajustement du layout
    fig.update_layout(
        xaxis=dict(title="Année"),
        yaxis=dict(title="Proportion de femmes"),
        width=900,
        height=500
    )

    fig.show()


In [ ]:
# Préparation des options du widget, ajout de l'option "all"
sports_names = sorted(df["sport"].dropna().unique())
options = ["all"] + list(sports_names)

# Création du widget Dropdown
sports_widget = widgets.Dropdown(
    options=options,
    description="Sports :",
    value="all"  # <- valeur par défaut sûre
)

# Création d'un output pour l'affichage du graphique
out = widgets.Output()

# Fonction de mise à jour du graphique lors du changement de valeur du widget
def update_graph(change=None):
    """
    Mise à jour du graphique interactif en fonction du sport sélectionné.
    """
    clear_output(wait=True)
    display(sports_widget)
    
    selected_sport = sports_widget.value
    
    if selected_sport == "all":
        plot_part_femmes(df, sport_code="all", sport_col="code_sport")
    else:
        # Conversion du nom du sport en code_sport
        codes = df.loc[df["sport"] == selected_sport, "code_sport"].unique()
        if len(codes) == 1:
            code = codes[0]
        else:
            code = list(codes)
        plot_part_femmes(df, sport_code=code, sport_col="code_sport")

# Liaison du widget à la fonction de mise à jour
sports_widget.observe(update_graph, names='value')

# Affichage initial
display(sports_widget, out)
update_graph()


In [ ]:
def plot_hand_heatmap(df_lic, sport_code="HAN", sport_col="code_sport"):
    """
    Affiche une heatmap interactive Plotly du nombre de licences par tranche d'âge et par année pour un sport donné.

    Paramètres
    ----------
    df_lic : pd.DataFrame
        DataFrame contenant au moins les colonnes ['annee', 'tranche_age', 'licences_annuelles', sport_col].
    sport_code : str
        Code du sport à filtrer (par défaut "HAN").
    sport_col : str
        Nom de la colonne contenant le code sport (par défaut 'code_sport').

    Output
    ------
    Heatmap interactive Plotly des licences par année et tranche d'âge.
    """
    df = df_lic.copy()

    # Filtrage par sport si demandé
    if sport_code != "all":
        df = df[df[sport_col] == sport_code]
        noms_sports = df["sport"].dropna().unique()
        titre_sport = noms_sports[0] if len(noms_sports) == 1 else ", ".join(noms_sports)
    else:
        titre_sport = "tous les sports"

    # Pivot : tranches d'âge en lignes, années en colonnes
    pivot = df.groupby(["tranche_age", "annee"])["licences_annuelles"] \
              .sum().reset_index() \
              .pivot(index="tranche_age", columns="annee", values="licences_annuelles")

    # Tracé heatmap Plotly
    fig = px.imshow(
        pivot,
        labels=dict(x="Année", y="Tranche d'âge", color="Licences"),
        x=pivot.columns,
        y=pivot.index,
        color_continuous_scale="Blues",
        aspect="auto"
    )

    # Inverser l'axe y pour que les plus jeunes soient en bas
    fig.update_yaxes(autorange="reversed")

    fig.update_layout(
        title=f"Heatmap Licences (année x tranche d’âge) - {titre_sport}",
        width=900,
        height=500
    )

    fig.show()


In [ ]:
# Préparation des options du widget
sports_names = sorted(df["sport"].dropna().unique())
options = ["all"] + list(sports_names)

# Création du widget Dropdown
sports_widget = widgets.Dropdown(
    options=options,
    description="Sports :",
    value="all"  # valeur par défaut sûre
)

# Output pour le graphique
out = widgets.Output()

# Fonction de mise à jour du graphique
def update_graph(change=None):
    """
    Mise à jour de la heatmap interactive en fonction du sport sélectionné.
    """
    clear_output(wait=True)
    display(sports_widget)
    
    selected_sport = sports_widget.value
    
    if selected_sport == "all":
        plot_hand_heatmap(df, sport_code="all", sport_col="code_sport")
    else:
        # Conversion du nom du sport en code_sport si besoin
        codes = df.loc[df["sport"] == selected_sport, "code_sport"].unique()
        if len(codes) == 1:
            code = codes[0]
        else:
            code = list(codes)
        plot_hand_heatmap(df, sport_code=code, sport_col="code_sport")

# Liaison du widget à la fonction de mise à jour
sports_widget.observe(update_graph, names='value')

# Affichage initial
display(sports_widget, out)
update_graph()
